## Script for Comparing Incorrectly Labeled Epochs

In [ ]:
import mne 
import os
import numpy as np 
import pandas as pd
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib import pyplot as plt
import tkinter as tk
import glob
import pickle
from sklearn.model_selection import train_test_split, KFold
import matplotlib
matplotlib.use('QtAgg') 

In [ ]:
# importing model 
with open("M2_all.pkl", "rb") as f:
    model = pickle.load(f)

In [ ]:
# importing features dataframe 
features_all = pd.read_pickle("training_features_19032026.pkl")

In [ ]:
X_zygo = features_all[["Zygo"]] 
X_corr = features_all[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

y_zygo = features_all["Num_Contractions_Zygo"].astype(int).to_numpy()
y_corr = features_all["Num_Contractions_Corr"].astype(int).to_numpy()

X = np.concatenate((X_zygo, X_corr), axis=0)
y = np.concatenate((y_zygo,y_corr),axis=0)       


indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42) # keep 20% purely for testing 

In [ ]:
n1 = len(X_zygo)
labels = np.where(idx_test < n1, "Zygo", "Corr")

idx_test_1 = idx_test[idx_test < n1]
idx_test_2 = idx_test[idx_test >= n1] - n1
X_test_zygo = X[idx_test_1]
X_test_corr = X[idx_test_2]
X_test = X[idx_test]

# get testing results 
y_pred_test_zygo = model.predict(X_test_zygo)   
y_pred_test_corr = model.predict(X_test_corr)   
y_pred_test  = model.predict(X_test)

y_pred_test_zygo = np.argmax(y_pred_test_zygo, axis=1)   
y_pred_test_corr = np.argmax(y_pred_test_corr, axis=1)   
y_pred_test = np.argmax(y_pred_test,axis=1)

# get other fields for dataframe 
subjects = features_all["Subject"] 
subject_subset = np.concatenate([subjects[idx_test_1],subjects[idx_test_2]])

epochs = features_all["Triggers_Order_Nap"] 
epochs_subset = np.concatenate([epochs[idx_test_1],epochs[idx_test_2]])

naps = features_all["Nap Number"] 
naps_subset = np.concatenate([naps[idx_test_1],naps[idx_test_2]])

# get mismatched epoch indices where the tested indices dont equal predicted 
epoch_idx = np.where(y[idx_test] != y_pred_test)

mismatched_subj = subject_subset[epoch_idx]
mismatched_epochs = epochs_subset[epoch_idx]
mismatched_naps = naps_subset[epoch_idx]

Pre-Processing

In [ ]:
# making dataframe for epoch rescoring 
rescore_epochs = pd.DataFrame({
    "Subject": mismatched_subj,
    "Nap Number": mismatched_naps,
    "Triggers_Order_Nap":mismatched_epochs,
    "Label": labels[epoch_idx],
    "Prediction": y_pred_test[epoch_idx]
}, index=epoch_idx[0])

rescore_epochs_reshape = (
    rescore_epochs
    .pivot_table(
        index=["Subject", "Nap Number", "Triggers_Order_Nap"],
        columns="Label",
        values="Prediction",
        aggfunc="first"
    )
    .rename(columns={
        "Zygo": "Prediction_Zygo",
        "Corr": "Prediction_Corr"
    })
    .reset_index()
)

In [ ]:
# making final dataframe for going through bad epochs 
rows = []
seen = set()

for _, row in rescore_epochs_reshape.iterrows():
    key = (row["Subject"], row["Nap Number"], row["Triggers_Order_Nap"])

    # skip duplicates
    if key in seen:
        continue
    seen.add(key)

    # find matching row(s) in the other dataframe
    match = features_all[
        (features_all["Subject"] == row["Subject"]) &
        (features_all["Nap Number"] == row["Nap Number"]) &
        (features_all["Triggers_Order_Nap"] == row["Triggers_Order_Nap"])
    ]

    # if there is a match, take the first one
    if not match.empty:
        match_row = match.iloc[0]

        # --- HANDLE NaNs HERE ---
        match_zygo = match_row["Zygo"]
        match_corr = match_row["Corr"]

        pred_zygo = row["Prediction_Zygo"]
        pred_corr = row["Prediction_Corr"]

        if pd.isna(pred_zygo):
            pred_zygo = np.argmax(model.predict(np.array(match_zygo.tolist()).reshape(1, 2251,1), verbose=0))

        if pd.isna(pred_corr):
            pred_corr = np.argmax(model.predict(np.array(match_corr.tolist()).reshape(1, 2251,1), verbose=0))


        rows.append({
            "Subject": row["Subject"],
            "Nap Number": row["Nap Number"],
            "Triggers_Order_Nap": row["Triggers_Order_Nap"],
            "Prediction_Zygo": pred_zygo,
            "Prediction_Corr": pred_corr,

            # examples of fields from df_other
            "Num_Contractions_Zygo": match_row["Num_Contractions_Zygo"],
            "Num_Contractions_Corr": match_row["Num_Contractions_Corr"],
            "Zygo": match_row["Zygo"],
            "Corr": match_row["Corr"],
        })

    else:
        # if no match exists, still keep the row
        rows.append({
            "Subject": row["Subject"],
            "Nap Number": row["Nap Number"],
            "Triggers_Order_Nap": row["Triggers_Order_Nap"],
            "Label": row["Label"],
            "Prediction_Zygo": pred_zygo,
            "Prediction_Corr": pred_corr,

            "Num_Contractions_Zygo": pd.NA,
            "Num_Contractions_Corr": pd.NA,
            "Zygo": pd.NA,
            "Corr": pd.NA
        })


mismatch_df = pd.DataFrame(rows)    


In [ ]:
mismatch_df.head(10)


Scoring for Mismatched Epochs

In [265]:
#GUI
frq = 250 
current_index=0
inter_trigger_length=10
window = 50
step = 1


resp_start_zygo = None
resp_end_zygo = None

resp_start_corr = None
resp_end_corr = None

contr_num_zygo = None
contr_num_corr = None

back = False 

epoch_seen = set() # tracking if epoch has been seen before 

df_triggers = pd.DataFrame(
    columns=[
    "Epoch",
    "Muscle_type",
    "Response_start_sample",
    "Response_end_sample",
    "Contraction_number"
])

def plot_figure(t):
    global frq 
    global mismatch_df,df_triggers
    global current_index, fig, mismatch_df,df_triggers,i,curr_muscle,contr_num_corr,contr_num_zygo
    global resp_start_zygo, resp_start_corr, resp_end_zygo, resp_end_corr, back 
    
 
    fig, ax = plt.subplots(2, 1) #, figsize=(10, 5))

    zygo = mismatch_df["Zygo"].iloc[t] 
    corr = mismatch_df["Corr"].iloc[t] 

    fig.suptitle(f"Epoch {t + 1}")
    print(epoch_seen)
    # title for corr plot 
    if (not back and (current_index+1 not in epoch_seen)):
        if contr_num_corr != None and resp_end_corr != None and resp_start_corr != None:
            fdct = {'color': 'g'}
            title = ax[0].set_title(f"SCORED : {contr_num_corr} CONTRACTIONS; START = {resp_start_corr}; END = {resp_end_corr}")
            title.set(**fdct)
        elif contr_num_corr ==None and resp_end_corr == None and resp_start_corr == None:
            fdct = {'color': 'r'}
            title = ax[0].set_title(f"SCORED : {contr_num_corr} CONTRACTIONS")
            title.set(**fdct)
        else:
            title = ax[0].set_title(f"UNFINISHED SCORING : {contr_num_corr} CONTRACTIONS; START = {resp_start_corr}; END = {resp_end_corr}")

        # title for zygo plot 
        if contr_num_zygo != None and resp_end_zygo != None and resp_start_zygo != None:
            fdct = {'color': 'g'}
            title = ax[1].set_title(f"SCORED : {contr_num_zygo} CONTRACTIONS; START = {resp_start_zygo}; END = {resp_end_zygo}")
            title.set(**fdct)
        elif contr_num_zygo ==None and resp_end_zygo == None and resp_start_zygo == None:
            fdct = {'color': 'r'}
            title = ax[1].set_title(f"SCORED : {contr_num_zygo} CONTRACTIONS")
            title.set(**fdct)
        else:
            title = ax[1].set_title(f"UNFINISHED SCORING : {contr_num_zygo} CONTRACTIONS; START = {resp_start_zygo}; END = {resp_end_zygo}")
    elif back or current_index+1 in epoch_seen: # could probably work as else 
        # define temporary variables for plotting 
        contr_num_corr_temp = df_triggers.loc[(df_triggers["Epoch"] == current_index+1) &
                            (df_triggers["Muscle_type"] == "Corr"),"Contraction_number"].iloc[0]
        
        contr_num_zygo_temp = df_triggers.loc[(df_triggers["Epoch"] == current_index+1) &
                            (df_triggers["Muscle_type"] == "Zygo"),"Contraction_number"].iloc[0]

        resp_start_corr_temp = df_triggers.loc[(df_triggers["Epoch"] == current_index+1) &
                            (df_triggers["Muscle_type"] == "Corr"),"Response_start_sample"].iloc[0]
        resp_end_corr_temp = df_triggers.loc[(df_triggers["Epoch"] == current_index+1) &
                            (df_triggers["Muscle_type"] == "Corr"),"Response_end_sample"].iloc[0]

        resp_start_zygo_temp = df_triggers.loc[(df_triggers["Epoch"] == current_index+1) &
                            (df_triggers["Muscle_type"] == "Zygo"),"Response_start_sample"].iloc[0]
        resp_end_zygo_temp = df_triggers.loc[(df_triggers["Epoch"] == current_index+1) &
                            (df_triggers["Muscle_type"] == "Zygo"),"Response_end_sample"].iloc[0]


        if contr_num_corr_temp != None and resp_end_corr_temp != None and resp_start_corr_temp != None:
            fdct = {'color': 'g'}
            title = ax[0].set_title(f"SCORED : {contr_num_corr_temp} CONTRACTIONS; START = {resp_start_corr_temp}; END = {resp_end_corr_temp}")
            title.set(**fdct)
        elif contr_num_corr ==None and resp_end_corr == None and resp_start_corr == None:
            fdct = {'color': 'r'}
            title = ax[0].set_title(f"SCORED : {contr_num_corr_temp} CONTRACTIONS")
            title.set(**fdct)
        else:
            title = ax[0].set_title(f"UNFINISHED SCORING : {contr_num_corr_temp} CONTRACTIONS; START = {resp_start_corr_temp}; END = {resp_end_corr_temp}")

        # title for zygo plot 
        if contr_num_zygo_temp != None and resp_end_zygo_temp != None and resp_start_zygo_temp != None:
            fdct = {'color': 'g'}
            title = ax[1].set_title(f"SCORED : {contr_num_zygo_temp} CONTRACTIONS; START = {resp_start_zygo_temp}; END = {resp_end_zygo_temp}")
            title.set(**fdct)
        elif contr_num_zygo_temp ==None and resp_end_zygo_temp == None and resp_start_zygo_temp == None:
            fdct = {'color': 'r'}
            title = ax[1].set_title(f"SCORED : {contr_num_zygo_temp} CONTRACTIONS")
            title.set(**fdct)
        else:
            title = ax[1].set_title(f"UNFINISHED SCORING : {contr_num_zygo_temp} CONTRACTIONS; START = {resp_start_zygo_temp}; END = {resp_end_zygo_temp}")

      
    # Corr subplot
    ax[0].plot(corr, color="blue", label="Corr")

    #ax[0].set_ylim(-ymax_emg, ymax_emg)

    ax[0].set_ylabel("Corr EMG [V]")
    ax[0].set_xlabel("Samples")

    # Zygo subplot
    ax[1].plot(zygo, label="Zygo",color="black")
    #ax[1].set_ylim(-ymax_emg, ymax_emg)

    ax[1].set_ylabel("Zygo EMG [V]")
    ax[1].set_xlabel("Samples")


    # Connect mouse click and key press events
    fig.canvas.mpl_connect('key_press_event', on_key)
    fig.canvas.mpl_connect('button_press_event', on_click)

    plt.tight_layout()
    plt.show()


def on_click(event):
    global current_index, fig, df_triggers,i, curr_muscle,contr_num_zygo,contr_num_corr
    global resp_start_zygo, resp_start_corr, resp_end_zygo, resp_end_corr, back 
    if event.inaxes:  # Check if click occurred in any axes

        curr_muscle = None
        for i, a in enumerate(event.canvas.figure.axes):
            if event.inaxes == a:
                curr_muscle = "Corr" if i == 0 else "Zygo"
                break


        x = int(event.xdata)

        if curr_muscle == "Zygo":
            if resp_start_zygo is None:
                resp_start_zygo = x
            elif resp_end_zygo is None:
                resp_end_zygo = x

        elif curr_muscle == "Corr":
            if resp_start_corr is None:
                resp_start_corr = x
            elif resp_end_corr is None:
                resp_end_corr = x

        plt.close()  # Close current figure
        plot_figure(current_index)         
             

# Keyboard press event handler
def on_key(event):
    global current_index, fig, mismatch_df,df_triggers,i,curr_muscle,contr_num_corr,contr_num_zygo
    global resp_start_zygo, resp_start_corr, resp_end_zygo, resp_end_corr, back 
    
    allowed_keys = {'0','1', '2', '3', '4', '5','6', '7', '8', '9'}
    key = event.key
        
    if key in allowed_keys:
        if curr_muscle == "Corr":
            contr_num_corr = int(key) 
            if ((resp_start_corr != None and resp_end_corr != None) or 
                contr_num_corr == 0):
                plt.close()
                plot_figure(current_index) 
        elif curr_muscle == "Zygo":
            contr_num_zygo = int(key) 
            if ((resp_start_zygo != None and resp_end_zygo != None) or
                contr_num_zygo == 0):
                plt.close()
                plot_figure(current_index) 
        
        
    elif event.key == 'right':  # Move to next figure
        current_index = (current_index + 1) % len(mismatch_df)  # Loop to the start
        # store in df and clear all vars 
        print(
        "Corr:", resp_start_corr, resp_end_corr,contr_num_corr,
        "| Zygo:", resp_start_zygo, resp_end_zygo,contr_num_zygo
    )
        
       
        row_zygo = {
        "Epoch": current_index,
        "Muscle_type": "Zygo",
        "Response_start_sample":resp_start_zygo,
        "Response_end_sample":resp_end_zygo,
        "Contraction_number":contr_num_zygo }

        row_corr = {
        "Epoch": current_index,
        "Muscle_type": "Corr",
        "Response_start_sample":resp_start_corr,
        "Response_end_sample":resp_end_corr,
        "Contraction_number": contr_num_corr}

        if current_index not in epoch_seen:
            df_triggers.loc[len(df_triggers)] = row_corr
            df_triggers.loc[len(df_triggers)] = row_zygo

        # update seen epochs 
        epoch_seen.add(current_index)

        df_triggers.head()
        # clear all vars 
        resp_start_zygo = None
        resp_end_zygo = None

        resp_start_corr = None
        resp_end_corr = None

        contr_num_zygo = None 
        contr_num_corr = None 

        # check if epoch already in data frame 
        back = False
        plt.close()  # Close current figure
        plot_figure(current_index)  # Plot the next figure

    elif event.key == 'left':  # Move to previous figure
        current_index = (current_index - 1) % len(mismatch_df)  # Loop to the end
        back = True 
        plt.close()  # Close current figure
        plot_figure(current_index)  # Plot the previous figure

    elif event.key == 'r':  # Move to previous figure
        '''
        df_triggers.loc[current_index, 'Response_start_sample'] = None
        df_triggers.loc[current_index, 'Response_end_sample'] = None 
        df_triggers.loc[current_index, 'Contraction_number'] = None
        df_triggers.loc[current_index, 'Muscle_type'] = None
        df_triggers.loc[current_index, 'RT_sec'] = None
        '''
        back = True 
        plt.close()  # Close current figure
        plot_figure(current_index)  # Plot the previous figure

    elif event.key == 'q':  # Custom action for specific key
        print("Quitting the plot!")
        plt.close()

    elif event.key == 'escape':  
        print("Quitting the plot!")
        plt.close(fig)  
        

# Plot the first figure
plot_figure(current_index)

set()
set()


QCoreApplication::exec: The event loop is already running


set()


QCoreApplication::exec: The event loop is already running


set()


...

{1, 2}


QCoreApplication::exec: The event loop is already running


Corr: 347 506 3 | Zygo: 529 857 3
{1, 2, 3}


QCoreApplication::exec: The event loop is already running


KeyboardInterrupt: 

In [266]:
df_triggers.head(10)

,Epoch,Muscle_type,Response_start_sample,Response_end_sample,Contraction_number
0,1,Corr,132,374,1
1,1,Zygo,205,442,1
2,2,Corr,446,1083,1
3,2,Zygo,522,814,2
4,3,Corr,347,506,3
5,3,Zygo,529,857,3
